# 19.2 A/B 测试分析 / A/B Test Analysis (t-test, Bootstrap, CUPED)

**中文**：19.1 讲了怎么**设计**实验(样本量、功效)。实验跑完、拿到数据后,**怎么分析**才能得出可信结论？本节讲三样核心工具:**① 假设检验(t 检验)** 给出 p 值和置信区间;**② Bootstrap** 用重采样给任意指标算置信区间(不靠正态假设);**③ CUPED** 用实验前的数据**免费降低方差**——同样的样本量,更窄的置信区间、更高的功效。CUPED 是微软、Booking、Netflix 等公司 A/B 平台的标配,也是面试加分项。
**English**: 19.1 covered how to **design** an experiment (sample size, power). After it runs and you have data, **how do you analyze it** for trustworthy conclusions? This section covers three core tools: **① hypothesis testing (t-test)** for p-values and confidence intervals; **② bootstrap** for CIs on any metric via resampling (no normality assumption); **③ CUPED** for **free variance reduction** using pre-experiment data — narrower CIs and higher power at the same sample size. CUPED is standard in the A/B platforms of Microsoft, Booking, Netflix, and a resume plus in interviews.

---

**中文**：**① 双样本 t 检验**:比较实验组与对照组的均值差是否"显著"(不是随机波动)。
**English**: **① Two-sample t-test**: is the difference in means between treatment and control "significant" (not random noise)?

$$t=\frac{\bar Y_B-\bar Y_A}{\sqrt{s_A^2/n_A+s_B^2/n_B}},\qquad \text{95\% CI}=(\bar Y_B-\bar Y_A)\pm z_{0.975}\cdot \text{SE}$$

**中文**：分子是两组均值差(观测到的效应),分母是标准误(SE,衡量这个差有多"稳")。**p 值**=假如真没差异,看到这么大(或更大)差异的概率;p<0.05 通常判为"显著"。**但更该看置信区间**——它同时告诉你效应的**方向、大小、和不确定性**。
**English**: The numerator is the difference in means (observed effect); the denominator is the standard error (SE, how "stable" that difference is). The **p-value** = probability of seeing a difference this large (or larger) if there were truly no difference; p<0.05 is usually deemed "significant." **But prefer the confidence interval** — it tells you the effect's **direction, magnitude, and uncertainty** at once.

**中文**：**② Bootstrap(自助法)**:t 检验假设数据近似正态。但很多指标(转化率、人均时长、点击/曝光比)不正态、甚至是**比值**。Bootstrap 不靠公式:**从样本里有放回地反复重采样**,每次算一遍指标差,重复几千次得到差异的经验分布——它的 2.5% 和 97.5% 分位数就是 95% 置信区间。**万能、不挑指标。**
**English**: **② Bootstrap**: the t-test assumes approximate normality. But many metrics (conversion rate, time-per-user, click/impression ratio) aren't normal or are even **ratios**. Bootstrap relies on no formula: **resample with replacement from the data repeatedly**, recompute the metric difference each time, thousands of times, to get an empirical distribution of the difference — its 2.5% and 97.5% quantiles are the 95% CI. **Universal, works for any metric.**

**中文**：**③ CUPED(用实验前数据控制方差)**:核心洞察——用户在实验前就有个**相关的历史指标**(如上周的活跃度)。用它把"实验后指标里本可预测的那部分波动"减掉,只留下实验真正引起的变化:
**English**: **③ CUPED (Controlled experiment Using Pre-Experiment Data)**: the key insight — users have a **correlated historical metric before the experiment** (e.g. last week's activity). Subtract the "predictable part" of the post-metric's fluctuation, leaving only what the experiment actually caused:

$$Y^{\text{cuped}}_i=Y_i-\theta\,(X_i-\bar X),\qquad \theta=\frac{\text{Cov}(Y,X)}{\text{Var}(X)}$$

**中文**：$X$ 是实验前的协变量。减去它的影响后,$Y^{\text{cuped}}$ 的**均值不变(无偏)但方差大幅下降**——方差**减少约 $\rho^2$**($\rho$=前后指标相关性)。相关性 0.7,方差就少一半,相当于**样本量翻倍却不花一分钱流量**。
**English**: $X$ is the pre-experiment covariate. After removing its influence, $Y^{\text{cuped}}$ has **the same mean (unbiased) but much lower variance** — variance drops by about **$\rho^2$** ($\rho$=pre/post correlation). At correlation 0.7, variance halves, equivalent to **doubling the sample size for free**.

> 💡 **面试速查 / Interview cheat-sheet（★★★ A/B分析必考）**
> **中文**：分析 A/B:**t/z 检验**给 p 值+CI(比较均值/比例), **优先看置信区间**(方向+大小+不确定性)而非只看 p<0.05。**Bootstrap**=重采样算 CI, 不挑指标(比值/分位数/非正态都行), 但慢。**CUPED**=用实验前协变量减方差(θ=Cov/Var), 无偏、方差降 ~ρ²——**免费提功效/缩样本**(工业标配)。**比值指标**(如人均点击=总点击/总用户, 分母也随机)要用 **Delta 方法**或 bootstrap 算方差, 不能当独立样本。**大坑**:①偷看膨胀假阳性;②多指标要 FDR 校正;③辛普森悖论(分层看)。
> **English**: Analyzing A/B: **t/z-test** gives p-value + CI (comparing means/proportions); **prefer the CI** (direction + magnitude + uncertainty) over just p<0.05. **Bootstrap** = resampling for CIs, works for any metric (ratios/quantiles/non-normal) but slow. **CUPED** = pre-experiment covariate to cut variance (θ=Cov/Var), unbiased, variance drops ~ρ² — **free power / smaller samples** (industry standard). **Ratio metrics** (e.g. clicks-per-user = total clicks / total users, random denominator) need the **delta method** or bootstrap for variance, not naive independent-sample formulas. **Big pitfalls**: ① peeking inflates false positives; ② FDR-correct for many metrics; ③ Simpson's paradox (check strata).


In [ ]:

# ============================================================
# 模拟一个 A/B 实验数据(含实验前协变量)/ simulate A/B data with a pre-experiment covariate
# 中文:两组各 n 用户。每人有"实验前指标"X(上周消费), 和"实验后指标"Y(本周消费)——二者相关。
#      实验组 Y 上多加了真实效应。我们要从数据里把这个效应估出来 + 给不确定性。
# English: n users per arm. Each has a pre-metric X (last week's spend) and post-metric Y (this week's),
#      correlated. Treatment adds a true effect to Y. We estimate that effect + its uncertainty.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
np.random.seed(1)
n=4000; true_effect=0.20; rho=0.7                             # 每组样本 / 真效应 / 前后相关性
X = np.random.randn(2*n)*1.0 + 5.0                           # 实验前协变量(上周消费)/ pre-covariate
Y_base = rho*(X-5.0) + np.random.randn(2*n)*np.sqrt(1-rho**2)# 实验后基线(与X相关)/ post baseline
group = np.r_[np.zeros(n), np.ones(n)].astype(int)          # 0=对照A, 1=实验B / control/treatment
Y = Y_base + true_effect*group                              # 实验组多加真效应 / add true effect to treatment
print(f"真实效应 / true effect: +{true_effect}, 前后指标相关性 ρ={rho}")
print(f"对照组均值 {Y[group==0].mean():.3f}, 实验组均值 {Y[group==1].mean():.3f}")


**中文**：先做**双样本 t 检验**,从零算 t 统计量、p 值、95% 置信区间,并与 scipy 对照验证。
**English**: First a **two-sample t-test** from scratch — compute the t-statistic, p-value, and 95% CI, and verify against scipy.


In [ ]:

# ============================================================
# ① 双样本 t 检验(从零)/ two-sample t-test from scratch
# ============================================================
a, b = Y[group==0], Y[group==1]
diff = b.mean() - a.mean()                                   # 观测效应 / observed effect
se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))   # 标准误 / standard error
t_stat = diff/se
p_value = 2*(1-stats.norm.cdf(abs(t_stat)))                 # 双侧 p 值 / two-sided p-value
ci = (diff-1.96*se, diff+1.96*se)                          # 95% 置信区间 / 95% CI
print(f"观测效应 / effect: {diff:.4f}  (真值 {true_effect})")
print(f"标准误 SE {se:.4f}, t={t_stat:.2f}, p={p_value:.2e}")
print(f"95% 置信区间 / CI: [{ci[0]:.4f}, {ci[1]:.4f}] (宽度 {ci[1]-ci[0]:.4f})")
# scipy 验证 / verify with scipy
t_sp, p_sp = stats.ttest_ind(b, a)
print(f"scipy 验证 / verify: t={t_sp:.2f} (ours {t_stat:.2f}), p={p_sp:.2e} → 一致")


**中文**：再做 **Bootstrap 置信区间**:反复有放回重采样、每次算一遍效应,得到效应的经验分布。它不依赖正态假设,对任意指标都适用。看它给出的 CI 和 t 检验是否一致(对均值差应该几乎一样)。
**English**: Now a **bootstrap confidence interval**: repeatedly resample with replacement and recompute the effect, yielding its empirical distribution. It assumes no normality and works for any metric. Check its CI matches the t-test (for a difference in means they should nearly coincide).


In [ ]:

# ============================================================
# ② Bootstrap 置信区间 / bootstrap CI
# ============================================================
def bootstrap_diff(a, b, B=5000):
    rng=np.random.default_rng(0); diffs=np.empty(B)
    for i in range(B):
        sa=rng.choice(a, len(a), replace=True)              # 有放回重采样对照组 / resample control
        sb=rng.choice(b, len(b), replace=True)              # 有放回重采样实验组 / resample treatment
        diffs[i]=sb.mean()-sa.mean()                        # 每次的效应 / effect each time
    return diffs
boot=bootstrap_diff(a,b)
ci_boot=(np.percentile(boot,2.5), np.percentile(boot,97.5))# 分位数即置信区间 / percentile CI
print(f"Bootstrap 95% CI: [{ci_boot[0]:.4f}, {ci_boot[1]:.4f}]")
print(f"t 检验     95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]  → 两者几乎一致(均值差时)")


**中文**：最后是**CUPED**:用实验前协变量 $X$ 把 $Y$ 里可预测的方差减掉。看它如何在**不改变效应估计**的前提下,**大幅缩小置信区间**(=提升功效)。
**English**: Finally **CUPED**: use the pre-experiment covariate $X$ to remove predictable variance from $Y$. See how it **shrinks the CI substantially** (= boosts power) **without changing the effect estimate**.


In [ ]:

# ============================================================
# ③ CUPED 方差削减 / CUPED variance reduction
# ============================================================
theta = np.cov(Y, X)[0,1]/np.var(X)                         # 最优系数 θ=Cov(Y,X)/Var(X)
Y_cuped = Y - theta*(X - X.mean())                          # CUPED 调整后的指标 / adjusted metric
ac, bc = Y_cuped[group==0], Y_cuped[group==1]
diff_cv = bc.mean()-ac.mean()
se_cv = np.sqrt(ac.var(ddof=1)/len(ac)+bc.var(ddof=1)/len(bc))
ci_cv = (diff_cv-1.96*se_cv, diff_cv+1.96*se_cv)
print(f"{'方法/method':<14}{'效应估计':>10}{'标准误 SE':>12}{'95% CI 宽度':>14}")
print(f"{'普通 t 检验':<14}{diff:>10.4f}{se:>12.4f}{ci[1]-ci[0]:>14.4f}")
print(f"{'CUPED':<14}{diff_cv:>10.4f}{se_cv:>12.4f}{ci_cv[1]-ci_cv[0]:>14.4f}")
print(f"\nCUPED 方差削减 / variance reduction: {1-(se_cv/se)**2:.0%}  (≈ ρ² = {rho**2:.0%})")
print(f"效应估计几乎不变(无偏), 但 CI 窄了 {(1-(ci_cv[1]-ci_cv[0])/(ci[1]-ci[0]))*100:.0f}% → 相当于免费加样本!")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(17,4.6))
# ① bootstrap 分布 + CI / bootstrap distribution
ax[0].hist(boot,bins=50,color="#4C72B0",alpha=0.7,density=True)
ax[0].axvline(ci_boot[0],color="r",ls="--"); ax[0].axvline(ci_boot[1],color="r",ls="--",label="95% CI")
ax[0].axvline(true_effect,color="g",lw=2,label="真效应 true")
ax[0].set_title("Bootstrap 效应分布 / bootstrap distribution"); ax[0].set_xlabel("效应估计"); ax[0].legend(fontsize=8)
# ② CUPED vs 普通:CI 对比 / CI comparison
ax[1].errorbar([0],[diff],yerr=1.96*se,fmt="o",color="#C44E52",capsize=8,ms=10,label="普通 t 检验")
ax[1].errorbar([1],[diff_cv],yerr=1.96*se_cv,fmt="o",color="#4C72B0",capsize=8,ms=10,label="CUPED")
ax[1].axhline(true_effect,color="g",ls="--",label="真效应"); ax[1].set_xticks([0,1]); ax[1].set_xticklabels(["普通","CUPED"])
ax[1].set_title("CUPED 缩小置信区间 / CUPED narrows the CI"); ax[1].set_ylabel("效应 ± 95%CI"); ax[1].legend(fontsize=8)
# ③ 方差削减 vs 相关性 / variance reduction vs correlation
rhos=np.linspace(0,0.95,20); reductions=[]
for r in rhos:
    Yb=r*(X-5)+np.random.randn(2*n)*np.sqrt(1-r**2); Yr=Yb+true_effect*group
    th=np.cov(Yr,X)[0,1]/np.var(X); Ycv=Yr-th*(X-X.mean())
    reductions.append(1-Ycv.var()/Yr.var())
ax[2].plot(rhos,reductions,"o-",color="#55A868",label="实测方差削减")
ax[2].plot(rhos,rhos**2,"k--",label="理论 ρ²")
ax[2].set_title("方差削减 ≈ ρ² / variance reduction ≈ ρ²"); ax[2].set_xlabel("前后指标相关性 ρ"); ax[2].set_ylabel("方差削减比例"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/ci02_viz.png",dpi=80); plt.show()
print("CUPED:前后指标越相关, 方差削减越多(≈ρ²), 白赚功效")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **置信区间比 p 值信息量大**:t 检验、bootstrap 都给出效应的区间估计。**优先报告置信区间**——它同时告诉你效应的方向、大小、不确定性。只说"p<0.05 显著"是不够的(一个大样本下微不足道的效应也能"显著",却毫无业务价值)。
2. **Bootstrap 万能但慢**:对均值差,bootstrap 的 CI 和 t 检验几乎一致(验证了两者正确)。bootstrap 的真正价值在**比值指标、分位数指标、非正态指标**——那些没有简洁公式的场景,重采样一律搞定。代价是计算慢(要重采样几千次)。
3. **CUPED 是"免费的功效"**:它把方差降了约 $\rho^2$(本例 ρ=0.7 → 降约 49%),**效应估计不变(无偏)、置信区间却窄了近三成**。这等价于"样本量凭空增加",在流量宝贵的大公司价值巨大。原理简单到优雅:**减掉实验前就能预测的那部分波动**。前后指标相关性越高,收益越大——所以选一个和主指标强相关的历史协变量是关键。诚实局限:①需要有实验前数据(新用户没有);②只减方差、不纠偏(它不是用来处理混杂的,随机化已经处理了混杂);③θ 要在合并数据上估(别分组估,会引入偏差)。

**English**:
1. **The CI is more informative than the p-value**: both the t-test and bootstrap give an interval estimate. **Prefer reporting the CI** — it conveys direction, magnitude, and uncertainty together. "p<0.05, significant" alone is insufficient (a trivially small effect can be "significant" at large sample sizes yet have no business value).
2. **Bootstrap is universal but slow**: for a difference in means, its CI nearly matches the t-test (verifying both). Bootstrap's real value is for **ratio metrics, quantile metrics, non-normal metrics** — cases with no neat formula, where resampling just works. The cost is compute (thousands of resamples).
3. **CUPED is "free power"**: it cuts variance by about $\rho^2$ (here ρ=0.7 → ~49% reduction), leaving the effect estimate unchanged (unbiased) while the CI narrows by nearly a third. This is equivalent to "sample size appearing out of thin air," hugely valuable where traffic is precious. The principle is elegantly simple: **subtract the fluctuation predictable before the experiment**. The higher the pre/post correlation, the bigger the gain — so choosing a historical covariate strongly correlated with the primary metric is key. Honest limits: ① needs pre-experiment data (new users have none); ② only reduces variance, doesn't remove bias (it's not for confounding — randomization already handled that); ③ estimate θ on pooled data (not per-group, which would bias it).

> 💼 **实战视角 / Practical angle**
> **中文**:分析 A/B 的实战:①**主指标看 CUPED 调整后的 CI**(几乎所有大厂 A/B 平台内置 CUPED / 回归调整);②**比值/人均类指标**用 delta 方法或 bootstrap 算方差(别把"人均"当独立样本, 分母也随机);③**多指标**做 FDR 校正、设护栏;④**分层看**(防辛普森悖论、找异质效应);⑤**别提前叫停**(偷看→假阳性, 要序贯检验)。CUPED 的推广=**回归调整/双重稳健**(用多个协变量甚至 ML 模型预测 Y 再残差化)。面试金句:*"报告效应的置信区间而非只报 p 值; 非正态/比值指标用 bootstrap; 用 CUPED(减去实验前可预测方差)白赚功效——方差降约 ρ²、估计仍无偏, 这是工业 A/B 的标准操作。"*
> **English**: Analyzing A/B in practice: ① **read the CUPED-adjusted CI for the primary metric** (nearly all big-tech A/B platforms build in CUPED / regression adjustment); ② **ratio / per-user metrics** need the delta method or bootstrap for variance (don't treat "per-user" as independent samples — the denominator is random too); ③ **many metrics** → FDR correction + guardrails; ④ **check strata** (against Simpson's paradox, and to find heterogeneous effects); ⑤ **don't stop early** (peeking → false positives; use sequential tests). CUPED generalizes to **regression adjustment / doubly-robust** (predict Y from several covariates or even an ML model, then residualize). Interview line: *"Report the effect's confidence interval, not just the p-value; use bootstrap for non-normal/ratio metrics; use CUPED (subtract pre-experiment predictable variance) for free power — variance drops ~ρ² and the estimate stays unbiased, a standard industrial A/B practice."*

---
### 小结 / Summary
- **中文**:t/z 检验给 p 值+CI(优先看 CI); bootstrap 重采样算任意指标的 CI(不挑分布)。
- **English**: t/z-test gives p-value + CI (prefer the CI); bootstrap resampling gives CIs for any metric (distribution-free).
- **中文**:CUPED 用实验前协变量减方差(降 ~ρ²), 无偏且缩窄 CI=免费提功效(工业标配)。
- **English**: CUPED uses a pre-experiment covariate to cut variance (~ρ²), unbiased and narrowing the CI = free power (industry standard).
- **中文**:比值指标用 delta/bootstrap; 优先报告置信区间; 注意偷看、多重比较、辛普森悖论。
- **English**: Ratio metrics need delta/bootstrap; prefer confidence intervals; mind peeking, multiple comparisons, Simpson's paradox.
